In [69]:
import sys; print(sys.executable)
import tensorflow as tf
from dense_pcn_layer import DensePCNLayer
from conv_pcn_layer import Conv2DPCNLayer
from transformer_pcn_layer import TransformerPCNLayer, PositionalEncodingLayer
%load_ext autoreload
%autoreload 2

c:\Users\user\miniconda3\envs\tf_py3.10\python.exe
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from urllib.request import urlretrieve
urlretrieve("http://images.cocodataset.org/zips/train2017.zip", "train2017.zip")
urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", "coco.zip")

('coco.zip', <http.client.HTTPMessage at 0x2b788ba3520>)

In [ ]:
import zipfile
zipfile.ZipFile('train2017.zip', 'r').extractall('train2017')
zipfile.ZipFile('coco.zip', 'r').extractall('coco')

In [5]:
import json
train_captions = json.load(open("coco/annotations/captions_train2017.json", 'r'))

In [6]:
import cv2
image_data = []
image_data_id = []
for image in train_captions['images'][:1000]:
    img = cv2.imread(f"train2017/train2017/{image['file_name']}")
    img = cv2.resize(img, (640, 640))
    image_data.append(img)
    image_data_id.append(image["id"])

In [7]:
import numpy as np
image_data = np.array(image_data)
image_data_id = np.array(image_data_id)

In [8]:
caption_data = []
caption_data_image_id =[]
for annotation in train_captions['annotations']:
    if (image_data_id == annotation['image_id']).sum() == 1:
        caption_data.append(annotation['caption'])
        caption_data_image_id.append(annotation['image_id'])
caption_data = np.array(caption_data)
caption_data_image_id = np.array(caption_data_image_id)

In [9]:
chars = set()
for caption in caption_data:
    for char in caption:
        chars.add(char)

In [10]:
chars.add('\0')
chars_arr = np.array(list(chars))

In [11]:
num_tokens = np.array([len(caption) for caption in caption_data]).max()

In [12]:
tokenized = []
for caption in caption_data:
    seq = []
    for i in range(num_tokens):
        try:
            seq.append(chars_arr == caption[i])
        except IndexError:
            seq.append(chars_arr == '\0')
    tokenized.append(seq)

In [13]:
tokenized_captions = np.array(tokenized)

In [14]:
caption_matching_images = []
for id in caption_data_image_id:
    caption_matching_images.append(image_data[image_data_id==id])
caption_matching_images = np.array(caption_matching_images)

In [16]:
np.where(chars_arr == '\0')

(array([33], dtype=int64),)

In [52]:
mask =  tf.where(tokenized_captions[:, :, 33], -1e9, 0.0) 

In [68]:
caption_matching_images = tf.convert_to_tensor(caption_matching_images[:2000])

In [71]:
import gc
gc.collect()

0

In [66]:
A = DensePCNLayer(128, 0.00001)
C = DensePCNLayer(200, 0.00001)
A.state = tf.Variable(tf.random.normal((2000, 185, 128)))
B = TransformerPCNLayer(3, 128, 4, 0.00001, A, [C], mask[:2000])
A.next_layers=[B]
C.prev_layer = B
b_out = B(A.state)
c_out = C(b_out)

In [31]:
for i in range(10):
    for layer in B.get_layers():
        layer.update_state()
        layer.update_wts()
        layer.update_b()
    print(tf.reduce_sum((B.predict_next() - B(A.predict_next()))**2)
        +tf.reduce_sum((C.predict_next() - C(B.predict_next()))**2)
        +tf.reduce_sum((A.predict_next()-B.predict_prev())**2)
        +tf.reduce_sum((B.predict_next()-C.predict_prev())**2))

tf.Tensor(20608116.0, shape=(), dtype=float32)
tf.Tensor(20452216.0, shape=(), dtype=float32)
tf.Tensor(20298226.0, shape=(), dtype=float32)
tf.Tensor(20146092.0, shape=(), dtype=float32)
tf.Tensor(19995764.0, shape=(), dtype=float32)
tf.Tensor(19847200.0, shape=(), dtype=float32)
tf.Tensor(19700360.0, shape=(), dtype=float32)
tf.Tensor(19555202.0, shape=(), dtype=float32)
tf.Tensor(19411692.0, shape=(), dtype=float32)
tf.Tensor(19269802.0, shape=(), dtype=float32)
